## はじめに

このノートブックでは、GlacierStyle ECサイトの各種テーブルにメタデータ情報を自動付与します。

**主な処理内容:**
- AI_GENERATE_TABLE_DESCによるテーブル・カラム説明の自動生成
- TRANSLATEによる日本語翻訳
- セマンティックビューの作成（Cortex Analyst向け）

In [ ]:
-- ============================================================================
-- 環境設定
-- ============================================================================
-- 使用するウェアハウスとスキーマを設定
USE WAREHOUSE COMPUTE_WH;
USE SCHEMA GLACIERSTYLE_DB.EC_ANALYTICS_SCHEMA;

## 3. 各種テーブルにメタデータ情報を付与

AI_GENERATE_TABLE_DESCとTRANSLATEを組み合わせて、テーブル・カラムに日本語コメントを自動付与します。

### 3-1. Description自動生成

ストアドプロシージャを使用して、テーブルとカラムの説明を自動生成します。

**処理フロー:**
1. AI_GENERATE_TABLE_DESC: テーブルデータを分析し、英語で説明を自動生成
2. SNOWFLAKE.CORTEX.TRANSLATE: 生成された説明を日本語に翻訳
3. ALTER TABLE/ALTER COLUMN: コメントとして設定

**対象テーブル:**
- ディメンジョンテーブル: dim_customers, dim_products
- ファクトテーブル: fact_orders, fact_payments, fact_web_logs
- Gold層テーブル: gold_sns_mentions_analyzed, gold_voice_logs, gold_ad_creative_analysis, gold_faq_documents, gold_operation_manuals, gold_sns_mentions_with_product_master

In [ ]:
-- ============================================================================
-- SPROC作成: テーブル・カラムコメント自動生成
-- ============================================================================
-- AI_GENERATE_TABLE_DESCでテーブル情報を分析し、TRANSLATEで日本語に翻訳
-- 生成された説明をテーブル・カラムのコメントとして設定
CREATE OR REPLACE PROCEDURE set_table_comments_ja(
    table_name_param STRING
)
RETURNS STRING
LANGUAGE PYTHON
RUNTIME_VERSION = '3.10'
PACKAGES = ('snowflake-snowpark-python')
HANDLER = 'main'
AS
$$
import json

def main(session, table_name_param):
    try:
        sql_generate = f"""
            CALL AI_GENERATE_TABLE_DESC(
                '{table_name_param}',
                OBJECT_CONSTRUCT('describe_columns', true, 'use_table_data', true)
            )
        """
        result = session.sql(sql_generate).collect()
        output = json.loads(result[0][0])
        
        table_info = output['TABLE'][0]
        table_desc_en = table_info['description']
        table_desc_en_escaped = table_desc_en.replace("'", "''")
        
        sql_translate_table = f"""
            SELECT SNOWFLAKE.CORTEX.TRANSLATE('{table_desc_en_escaped}', 'en', 'ja')
        """
        table_desc_ja_result = session.sql(sql_translate_table).collect()
        table_desc_ja = table_desc_ja_result[0][0]
        table_desc_ja_escaped = table_desc_ja.replace("'", "''")
        
        sql_set_table_comment = f"ALTER TABLE {table_name_param} SET COMMENT = '{table_desc_ja_escaped}'"
        session.sql(sql_set_table_comment).collect()
        
        columns = output.get('COLUMNS', [])
        for column in columns:
            column_name = column['name']
            column_desc_en = column['description']
            column_desc_en_escaped = column_desc_en.replace("'", "''")
            
            sql_translate_col = f"""
                SELECT SNOWFLAKE.CORTEX.TRANSLATE('{column_desc_en_escaped}', 'en', 'ja')
            """
            column_desc_ja_result = session.sql(sql_translate_col).collect()
            column_desc_ja = column_desc_ja_result[0][0]
            column_desc_ja_escaped = column_desc_ja.replace("'", "''")
            
            sql_set_col_comment = f"ALTER TABLE {table_name_param} ALTER COLUMN {column_name} SET COMMENT '{column_desc_ja_escaped}'"
            session.sql(sql_set_col_comment).collect()
        
        return f'SUCCESS: {table_name_param}'
    except Exception as e:
        return f'ERROR: {str(e)}'
$$;

In [ ]:
-- ============================================================================
-- 各テーブルにメタデータコメントを付与
-- ============================================================================
-- ディメンジョンテーブル
CALL set_table_comments_ja('dim_customers');
CALL set_table_comments_ja('dim_products');

-- ファクトテーブル
CALL set_table_comments_ja('fact_orders');
CALL set_table_comments_ja('fact_payments');
CALL set_table_comments_ja('fact_web_logs');

-- Gold層テーブル
CALL set_table_comments_ja('gold_sns_mentions_analyzed');
CALL set_table_comments_ja('gold_voice_logs');
CALL set_table_comments_ja('gold_ad_creative_analysis');
CALL set_table_comments_ja('gold_faq_documents');
CALL set_table_comments_ja('gold_operation_manuals');
CALL set_table_comments_ja('gold_sns_mentions_with_product_master');

In [ ]:
-- ============================================================================
-- メタデータ付与結果の確認
-- ============================================================================
-- テーブル・カラムに設定されたコメントを確認
DESC TABLE fact_orders;

### 3-2. セマンティックビュー作成

Cortex Analyst向けのセマンティックビューを作成します。

**セマンティックビューとは:**
- ビジネスメトリクスやエンティティの関係を定義
- 自然言語クエリ（Text-to-SQL）を可能にする
- ビジネス用語の同義語（シノニム）を定義

**作成方法:**
- オプション1: Snowsight UI上でウィザードを使用
- オプション2: SQLクエリで直接作成

### オプション1. Snowsight上での作成

1. 次のいずれかの方法で、セマンティックビューを作成するためのウィザードにアクセスします。

`1-1. データベースオブジェクトエクスプローラー`:
- Snowsight にサインインします。
- ナビゲーションメニューで Catalog » Database Explorer を選択します。
- セマンティックビューを作成するデータベースとスキーマを選択します。
- Create » Semantic View » Create with guided setup を選択します。

`1-2. Cortex Analyst`:
- Snowsight にサインインします。
- ナビゲーションメニューで AI & ML » Cortex Analyst を選択します。
- Create new » Create new Semantic View を選択します。


2. ウィザードの Getting started ステップで次を実行します。
- Location to store から、モデルを格納するデータベースとスキーマを選択します。
- Name に、セマンティックビューの名前を入力します。
  - 文字またはアンダースコアで始まり、文字、数字、アンダースコア、ドル記号のみを含む名前を指定する必要があります。
- （オプション） Description で、セマンティックビューによって利用可能になるデータの説明をします。
- Next を選択します。


3. `ウィザードの Select tables ステップで`:
- All タブで、セマンティックビューで使用するデータを含むテーブルまたはビューを選択します。次の点に注意してください。
  - 少なくとも1つのテーブルまたは表示を選択する必要があります。
  - パフォーマンスを向上させるために、10以上のテーブルを選択しないでください。
  - 選択したテーブルとビューのリストを表示するには、 Selected タブを選択します。
- Next を選択します。


4. `ウィザードの Select columns ステップで:`
- ビューに含める列を選択します。
  - テーブルまたは表示内のすべての列を選択するには、テーブルまたはビューを選択します。
  - パフォーマンス向上のため、50列以上は選択しないでください。
- Create and Save を選択します。


5. `Logical tables 中`:
- 各テーブルまたはビューに定義されているファクト、ディメンジョン、およびメトリクスを確認します。
- ビジネスに適した名称と説明を提供します。
- 必要なファクト、ディメンジョン、メトリクスを追加します。

6. `Relationships 中`:
- ジェネレーターによって定義された関係を確認します。
- 必要に応じてリレーションシップのプロパティを変更します。
- 必要なリレーションシップを追加してください。
- セマンティックビューに変更を加えた場合は、Save を選択します。

7. セマンティックビューに変更を加えた場合は、Save を選択します。

### オプション2. SQLクエリでの作成

In [ ]:
-- ============================================================================
-- GlacierStyle EC分析用セマンティックビューの作成
-- ============================================================================
-- Cortex Analyst向けにテーブル関係・メトリクス・ディメンションを定義
CREATE OR REPLACE SEMANTIC VIEW glacierstyle_db.ec_analytics_schema.ec_analysis_semantic_view
    COMMENT = 'GlacierStyle ECサイト分析用セマンティックビュー'

    TABLES (
        customers AS glacierstyle_db.ec_analytics_schema.dim_customers
            PRIMARY KEY (customer_id)
            WITH SYNONYMS ('顧客', 'カスタマー', '会員')
            COMMENT = '顧客マスタテーブル',

        products AS glacierstyle_db.ec_analytics_schema.dim_products
            PRIMARY KEY (product_id)
            WITH SYNONYMS ('商品', 'プロダクト', 'アイテム')
            COMMENT = '商品マスタテーブル',

        orders AS glacierstyle_db.ec_analytics_schema.fact_orders
            PRIMARY KEY (order_id)
            WITH SYNONYMS ('注文', 'オーダー', '受注')
            COMMENT = '注文トランザクションテーブル',

        payments AS glacierstyle_db.ec_analytics_schema.fact_payments
            PRIMARY KEY (payment_id)
            WITH SYNONYMS ('決済', '支払い', 'ペイメント')
            COMMENT = '決済トランザクションテーブル',

        web_logs AS glacierstyle_db.ec_analytics_schema.fact_web_logs
            PRIMARY KEY (log_id)
            WITH SYNONYMS ('アクセスログ', 'Webログ', '行動ログ')
            COMMENT = 'Webサイトアクセスログテーブル'
    )

    RELATIONSHIPS (
        orders_to_customers AS orders (customer_id) REFERENCES customers,
        orders_to_products AS orders (product_id) REFERENCES products,
        payments_to_orders AS payments (order_id) REFERENCES orders,
        web_logs_to_customers AS web_logs (customer_id) REFERENCES customers,
        web_logs_to_products AS web_logs (product_id) REFERENCES products
    )

    FACTS (
        orders.order_amount AS total_amount COMMENT = '注文金額',
        orders.order_quantity AS quantity COMMENT = '注文数量',
        orders.order_discount AS discount_amount COMMENT = '割引金額',
        payments.pay_amount AS payment_amount COMMENT = '決済金額',
        web_logs.time_spent AS time_on_page COMMENT = 'ページ滞在時間'
    )

    DIMENSIONS (
        customers.customer_name AS CONCAT(first_name, ' ', last_name) WITH SYNONYMS = ('顧客名') COMMENT = '顧客氏名',
        customers.gender WITH SYNONYMS = ('性別') COMMENT = '性別',
        customers.membership_tier WITH SYNONYMS = ('会員ランク') COMMENT = '会員ランク',
        customers.prefecture WITH SYNONYMS = ('都道府県') COMMENT = '都道府県',
        products.product_name WITH SYNONYMS = ('商品名') COMMENT = '商品名',
        products.category_l1 WITH SYNONYMS = ('大カテゴリ') COMMENT = '商品カテゴリ（大分類）',
        products.category_l2 WITH SYNONYMS = ('中カテゴリ') COMMENT = '商品カテゴリ（中分類）',
        orders.order_date AS DATE(order_datetime) WITH SYNONYMS = ('注文日') COMMENT = '注文日',
        orders.order_year AS YEAR(order_datetime) WITH SYNONYMS = ('注文年') COMMENT = '注文年',
        orders.order_month AS MONTH(order_datetime) WITH SYNONYMS = ('注文月') COMMENT = '注文月',
        orders.order_status WITH SYNONYMS = ('注文ステータス') COMMENT = '注文ステータス',
        orders.channel AS order_channel WITH SYNONYMS = ('チャネル') COMMENT = '販売チャネル',
        orders.pay_method AS payment_method WITH SYNONYMS = ('支払方法') COMMENT = '支払方法',
        web_logs.event_type WITH SYNONYMS = ('イベント種別') COMMENT = 'イベント種別',
        web_logs.page_category WITH SYNONYMS = ('ページ種別') COMMENT = 'ページカテゴリ',
        web_logs.device_type WITH SYNONYMS = ('デバイス') COMMENT = 'デバイス種別',
        web_logs.traffic_source AS utm_source WITH SYNONYMS = ('流入元') COMMENT = '流入元'
    )

    METRICS (
        total_revenue AS SUM(orders.order_amount) ON orders WITH SYNONYMS = ('売上', '総売上') COMMENT = '総売上金額',
        order_count AS COUNT(DISTINCT order_id) ON orders WITH SYNONYMS = ('注文件数') COMMENT = '注文件数',
        avg_order_value AS AVG(orders.order_amount) ON orders WITH SYNONYMS = ('平均注文金額', 'AOV') COMMENT = '平均注文金額',
        total_quantity AS SUM(orders.order_quantity) ON orders WITH SYNONYMS = ('総販売数') COMMENT = '総販売数量',
        total_discount AS SUM(orders.order_discount) ON orders WITH SYNONYMS = ('総割引額') COMMENT = '総割引金額',
        customer_count AS COUNT(DISTINCT customer_id) ON customers WITH SYNONYMS = ('顧客数') COMMENT = '顧客数',
        session_count AS COUNT(DISTINCT session_id) ON web_logs WITH SYNONYMS = ('セッション数') COMMENT = 'セッション数'
    );

In [ ]:
-- ============================================================================
-- セマンティックビューの確認
-- ============================================================================
-- 作成したセマンティックビューの定義を確認
DESCRIBE SEMANTIC VIEW ec_analysis_semantic_view;

## まとめ

このノートブックでは、GlacierStyle ECサイトの各種テーブルにメタデータ情報を自動付与しました。

### 実施した処理

**3-1. テーブル・カラムコメントの自動生成**
- AI_GENERATE_TABLE_DESCでテーブル構造とデータを分析し説明を生成
- TRANSLATEで日本語に翻訳
- ALTER TABLE/ALTER COLUMNでコメントとして設定

**3-2. セマンティックビューの作成**
- Cortex Analyst向けのec_analysis_semantic_viewを作成
- TABLES: 5つのベーステーブル（customers, products, orders, payments, web_logs）
- RELATIONSHIPS: テーブル間の関係を定義
- FACTS/DIMENSIONS/METRICS: ビジネス指標と分析軸を定義
- SYNONYMS: 日本語の同義語を設定

### 対象テーブル一覧

| 層 | テーブル名 | 説明 |
|---|---|---|
| Dimension | dim_customers | 顧客マスタ |
| Dimension | dim_products | 商品マスタ |
| Fact | fact_orders | 注文トランザクション |
| Fact | fact_payments | 決済トランザクション |
| Fact | fact_web_logs | Webアクセスログ |
| Gold | gold_sns_mentions_analyzed | SNS分析結果 |
| Gold | gold_voice_logs | 音声ログ分析結果 |
| Gold | gold_ad_creative_analysis | 広告クリエイティブ分析結果 |
| Gold | gold_faq_documents | FAQドキュメント |
| Gold | gold_operation_manuals | 運用マニュアル |
| Gold | gold_sns_mentions_with_product_master | SNS×商品マスタ突合結果 |

### 使用したCortex AI関数

- `AI_GENERATE_TABLE_DESC`: テーブル・カラム説明の自動生成
- `TRANSLATE`: 多言語翻訳（英語→日本語）

### 次のステップ

- Cortex Analystを使用した自然言語クエリの実行
- 各種AIエージェントの構築